In [ ]:
FILE_NAME='data/html/sample-html.html'

# Parsowanie HTML
W tym notatniku porównujemy kilka popularnych sposobów zamieniania surowego HTML na ustrukturyzowane fragmenty, które można przekazać do dalszych etapów przetwarzania.
1. **BeautifulSoup** - lekki parser idealny do szybkiego podglądu lub prostego wyodrębniania danych.
2. **Unstructured.io** - biblioteka do dzielenia dokumentów, która automatycznie segmentuje treść na elementy z typami.
3. Inne biblioteki i frameworki do parsowania dokumentów HTML.

## BeautifulSoup

**Beautiful Soup** to biblioteka Pythona zaprojektowana do szybkiego web scrapingu i screen scrapingu. Od 2004 roku pomaga deweloperom wyodrębniać dane ze słabo ustrukturyzowanych dokumentów HTML i XML przy minimalnej ilości kodu. Do najważniejszych funkcji należą:

*   **Proste metody nawigacji i wyszukiwania** do parsowania i modyfikowania drzew dokumentu
*   **Automatyczna obsługa kodowania**, konwertująca wejście do Unicode i wyjście do UTF-8
*   **Elastyczność parserów**, współpraca z `lxml`, `html5lib` i wbudowanym parserem Pythona
*   **Mocne możliwości wyszukiwania**, takie jak znajdowanie linków po klasie, dopasowywanie adresów URL lub wyodrębnianie zagnieżdżonych elementów

Beautiful Soup jest lekki, łatwy do integracji i szeroko używany zarówno w projektach prywatnych, jak i firmowych. Jest dostępny przez PyPI (`pip install beautifulsoup4`) i obsługuje Python 3.7+. Na licencji MIT idealnie nadaje się dla deweloperów, którzy potrzebują szybkiego i niezawodnego wyodrębniania danych z niechlujnych stron internetowych.

In [ ]:
from bs4 import BeautifulSoup

# Parsujemy plik HTML i pokazujemy najczęstsze elementy
with open(FILE_NAME, 'r', encoding='utf-8') as f:
    soup = BeautifulSoup(f, 'html.parser')

title = soup.title.get_text(strip=True) if soup.title else 'N/A'
print(f'Tytuł dokumentu: {title}\n')

headings = [(tag.name.upper(), tag.get_text(' ', strip=True)) for tag in soup.find_all(['h1', 'h2', 'h3'])]
if headings:
    print('Nagłówki:')
    for level, text in headings:
        print(f' - {level}: {text}')
else:
    print('Nie znaleziono nagłówków.')

links = [
    (link.get_text(' ', strip=True) or '(brak tekstu)', link['href'])
    for link in soup.find_all('a', href=True)
    if not link['href'].startswith('#')
 ]
print(f"\nZnaleziono {len(links)} link(ów):")
for idx, (text, href) in enumerate(links, start=1):
    print(f' {idx}. {text} -> {href}')

paragraphs = [p.get_text(' ', strip=True) for p in soup.find_all('p')]
if paragraphs:
    print(f"\nFragment pierwszego akapitu: {paragraphs[0][:120]}...")

table = soup.find('table')
if table:
    rows = [
        [cell.get_text(' ', strip=True) for cell in row.find_all(['th', 'td'])]
        for row in table.find_all('tr')
    ]
    print('\nPodgląd pierwszej tabeli:')
    for row in rows[:3]:
        print(' | '.join(row))
else:
    print('\nNie wykryto tabeli.')

## unstructured.io

**Unstructured.io** to otwartoźródłowa biblioteka ETL zaprojektowana do konwersji złożonych dokumentów - takich jak PDF-y, pliki Word, HTML i obrazy - do czystych, ustrukturyzowanych danych zoptymalizowanych pod kątem użycia z dużymi modelami językowymi (LLM). Udostępnia modułowe komponenty do:

*   **wczytywania dokumentów i wstępnego przetwarzania**
*   **automatycznego dzielenia na partie i wykrywania formatu**
*   **wzbogacania tabel i obrazów**
*   **dzielenia na fragmenty i generowania embeddingów**

Unstructured obsługuje zarówno lokalne środowiska deweloperskie, jak i wdrożenia kontenerowe przez Docker. Łatwo integruje się z Pythonem i oferuje konektory dla platform takich jak Discord. Biblioteka jest dobrze dopasowana do budowania skalowalnych, produkcyjnych potoków danych i jest dostępna przez PyPI oraz GitHub.

In [ ]:

from unstructured.partition.html import partition_html
elements = partition_html(filename=FILE_NAME, infer_table_structure=True)
print(f"Przeanalizowano {len(elements)} element(ów) z dokumentu HTML.")

In [ ]:
from collections import Counter
from textwrap import shorten

type_counts = Counter(element.category for element in elements)
print('Liczba elementów według kategorii:')
for category, count in type_counts.most_common():
    print(f' - {category}: {count}')

print('\nPrzykładowe elementy:')
for i, element in enumerate(elements[:5], start=1):
    if element.category == 'Table':
        chunk_text = element.metadata.text_as_html or ''
    else:
        chunk_text = element.text or ''
    snippet = shorten(chunk_text, width=100, placeholder='…')
    page_number = getattr(element.metadata, 'page_number', None)
    source = getattr(element.metadata, 'filename', None)
    metadata_bits = [f'strona {page_number}' if page_number else None, source]
    metadata_str = ' | '.join(bit for bit in metadata_bits if bit) or 'brak metadanych'
    print(f"{i}. [{element.category}] {snippet}")
    print(f"    metadane: {metadata_str}")